In [ ]:
# %% [markdown]
# # 02 - Preprocessing
# **Module:** Deep Learning (MOD006565)
# **Description:** Load saved splits, define transforms and augmentation, build DataLoaders, visualise samples.

# %% [markdown]
# ## Imports

# %%
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image

# %% [markdown]
# ## Configuration

# %%
SPLIT_DIR  = '/workspace/food101-dl-project/splits'
OUTPUT_DIR = '/workspace/food101-dl-project/outputs'
BATCH_SIZE = 64
IMAGE_SIZE = 128

os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# %% [markdown]
# ## Load Class Mapping

# %%
with open(f'{SPLIT_DIR}/class_mapping.json', 'r') as f:
    mapping = json.load(f)

SELECTED_CLASSES = mapping['selected_classes']
NUM_CLASSES      = len(SELECTED_CLASSES)

print(f"Classes loaded : {NUM_CLASSES}")

# %% [markdown]
# ## Load Splits from Disk

# %%
print("Loading splits from disk...")

train_images = np.load(f'{SPLIT_DIR}/train_images.npy')
train_labels = np.load(f'{SPLIT_DIR}/train_labels.npy')
val_images   = np.load(f'{SPLIT_DIR}/val_images.npy')
val_labels   = np.load(f'{SPLIT_DIR}/val_labels.npy')
test_images  = np.load(f'{SPLIT_DIR}/test_images.npy')
test_labels  = np.load(f'{SPLIT_DIR}/test_labels.npy')

print(f"Train      : {len(train_images)} images")
print(f"Validation : {len(val_images)} images")
print(f"Test       : {len(test_images)} images")

# %% [markdown]
# ## Custom Dataset Class

# %%
class Food101Dataset(Dataset):
    """
    Custom PyTorch Dataset for the Food-101 subset.
    Accepts pre-split numpy arrays and applies transforms on load.
    """
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = Image.fromarray(self.images[idx].astype(np.uint8))
        label = int(self.labels[idx])
        if self.transform:
            image = self.transform(image)
        return image, label

# %% [markdown]
# ## Define Transforms
#
# Training transforms include augmentation to improve generalisation:
# - Random crop (resize to 140 first, then crop to 128) introduces spatial variation
# - Horizontal flip simulates mirror images of food
# - Colour jitter accounts for lighting and camera variation
#
# Validation and test transforms apply only deterministic resizing and normalisation
# to ensure consistent and unbiased evaluation.
#
# Normalisation uses ImageNet mean and standard deviation, consistent with the
# DeepFood paper which fine-tunes from an ImageNet pretrained model.

# %%
train_transform = transforms.Compose([
    transforms.Resize((140, 140)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Train transform    : Resize(140) -> RandomCrop(128) -> HFlip -> ColorJitter -> Normalize")
print("Val/Test transform : Resize(128) -> Normalize")

# %% [markdown]
# ## Build Datasets and DataLoaders

# %%
train_dataset = Food101Dataset(train_images, train_labels, transform=train_transform)
val_dataset   = Food101Dataset(val_images,   val_labels,   transform=val_test_transform)
test_dataset  = Food101Dataset(test_images,  test_labels,  transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Batch size         : {BATCH_SIZE}")
print(f"Train batches      : {len(train_loader)}")
print(f"Validation batches : {len(val_loader)}")
print(f"Test batches       : {len(test_loader)}")

# %% [markdown]
# ## Visualise Sample Images

# %%
def denormalise(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = tensor.numpy().transpose((1, 2, 0))
    img  = std * img + mean
    return np.clip(img, 0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(denormalise(images[i]))
    ax.set_title(SELECTED_CLASSES[labels[i]], fontsize=7)
    ax.axis('off')

plt.suptitle('Sample Training Images after Augmentation (128x128)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Sample image grid saved to {OUTPUT_DIR}/sample_images.png")

# %% [markdown]
# ## Class Distribution

# %%
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)

plt.figure(figsize=(18, 4))
plt.bar(range(NUM_CLASSES), class_counts, color='steelblue')
plt.xticks(range(NUM_CLASSES), SELECTED_CLASSES, rotation=90, fontsize=7)
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.title('Training Set Class Distribution')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Class distribution plot saved to {OUTPUT_DIR}/class_distribution.png")

# %% [markdown]
# ## Summary

# %%
print("Preprocessing Summary")
print(f"Image size         : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch size         : {BATCH_SIZE}")
print(f"Train samples      : {len(train_dataset)}")
print(f"Validation samples : {len(val_dataset)}")
print(f"Test samples       : {len(test_dataset)}")
print(f"Num classes        : {NUM_CLASSES}")
print("DataLoaders are ready for training.")